In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Clone your GitHub repo
!git clone https://github.com/emmadionne1/Multimodal-Machine-Learning-Project.git
%cd Multimodal-Machine-Learning-Project


Cloning into 'Multimodal-Machine-Learning-Project'...
remote: Enumerating objects: 196, done.
remote: Total 196 (delta 0), reused 0 (delta 0), pack-reused 196 (from 1)
Receiving objects: 100% (196/196), 862.84 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (85/85), done.
Updating files: 100% (98/98), done.
/content/Multimodal-Machine-Learning-Project


In [ ]:
!pip install torch transformers datasets accelerate timm pillow tqdm numpy sentencepiece evaluate config rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=99bec0ff34a46acf056e8e0345434cdd6d5a7c47e04c49aafec07516400dc544
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
%cd Multimodal-Machine-Learning-Project


/content/Multimodal-Machine-Learning-Project


In [ ]:

import torch
import re
import numpy as np
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset

from base_architecture import (
    load_vision_model, load_language_model,
    Qwen3VLProjector, CustomVLM, DEVICE
)


DEVICE = "cuda"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
MODEL_VISION = "google/siglip2-so400m-patch16-512"
MODEL_LLM = "Qwen/Qwen3-4B-Instruct-2507"

PROJECTOR_PATH = "/content/drive/MyDrive/Multimodal Project/pretraining weights/7e3_a80_mediumishish_batch.pt"
#"/content/Multimodal-Machine-Learning-Project/outputs/final_run_full_dataset_A40_proj_final_weights.pt"  # 🔥 UPLOAD YOUR .pt FILE HERE

# DATASET_NAME = "chiragtubakad/chart-to-table"
MAX_NEW_TOKENS = 512

PROMPT = "What is 2 + 2?" # "You are a chart summarization assistant. Describe the chart clearly"

In [ ]:
# Load vision model
vision_model, processor = load_vision_model(MODEL_VISION)
v_conf = vision_model.config.vision_config if hasattr(vision_model.config, "vision_config") else vision_model.config
vision_dim = v_conf.hidden_size

# Load language model
language_model, tokenizer = load_language_model(MODEL_LLM)
llm_dim = language_model.config.hidden_size

# Add image token if missing
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
    language_model.resize_token_embeddings(len(tokenizer))

image_token_id = tokenizer.convert_tokens_to_ids("<image>")

# 🔥 Load YOUR projector
projector = Qwen3VLProjector(vision_dim, llm_dim, 2).to(DEVICE, dtype=torch.bfloat16)
projector.load_state_dict(torch.load(PROJECTOR_PATH, map_location="cpu"))

vision_model = vision_model.to(DEVICE, dtype=torch.bfloat16)
language_model = language_model.to(DEVICE, dtype=torch.bfloat16)
projector = projector.to(DEVICE, dtype=torch.bfloat16)

# Build full model
model = CustomVLM(
    vision_model,
    processor,
    language_model,
    tokenizer,
    projector,
    image_token_id
)
model = model.to(DEVICE)
model.eval()
print("✅ Model loaded successfully")

Loading Vision: google/siglip2-so400m-patch16-512


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ Model loaded successfully


In [ ]:
ds = load_dataset("HuggingFaceM4/the_cauldron", "chart2text")["train"]
print("Dataset size:", len(ds))

README.md: 0.00B [00:00, ?B/s]

chart2text/train-00000-of-00003-3a2ec464(…):   0%|          | 0.00/365M [00:00<?, ?B/s]

chart2text/train-00001-of-00003-a65d1189(…):   0%|          | 0.00/361M [00:00<?, ?B/s]

chart2text/train-00002-of-00003-8626ac7f(…):   0%|          | 0.00/378M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26961 [00:00<?, ? examples/s]

Dataset size: 26961


In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s\.]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def relaxed_match(pred, gt):
    pred_norm = normalize_text(pred)
    gt_norm = normalize_text(gt)

    if pred_norm == gt_norm or gt_norm in pred_norm:
        return True

    try:
        pv = float(re.findall(r"-?\d+\.?\d*", pred)[0])
        gv = float(re.findall(r"-?\d+\.?\d*", gt)[0])
        if abs(pv - gv) / max(abs(gv), 1e-6) <= 0.05:
            return True
    except:
        pass

    return False

def f1_score(pred_list, gt_list):
    matched_gt = set()
    matched_pred = set()

    for i, p in enumerate(pred_list):
        for j, g in enumerate(gt_list):
            if j in matched_gt:
                continue
            if relaxed_match(p, g):
                matched_pred.add(i)
                matched_gt.add(j)
                break

    precision = len(matched_pred) / max(len(pred_list), 1)
    recall = len(matched_gt) / max(len(gt_list), 1)
    return 2 * precision * recall / max(precision + recall, 1e-6)

def levenshtein(a, b):
    a, b = str(a), str(b)
    if len(a) < len(b): a, b = b, a
    if not b: return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

In [ ]:
exact_correct = 0
relaxed_correct = 0
rms_f1_list = []
lev_list = []
total = 0
from rouge_score import rouge_scorer
import evaluate

bleu = evaluate.load("bleu")
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
bleu_preds = []
bleu_refs = []
rouge_scores = []

# limit first for safety
ds_eval = ds.select(range(2700))  # 🔥 increase later

for sample in tqdm(ds_eval):

    # ---- Prompt ----
    messages = [{"role": "user", "content": "<image>\n" + PROMPT}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # ---- Image ----
    image = sample["images"][0]
    if image.mode != "RGB":
      image = image.convert("RGB")

    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.bfloat16)

    # ---- Text ----
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    attention_mask = torch.ones_like(inputs.input_ids).to(DEVICE)

    # ---- Generate ----
    with torch.no_grad():
        pred = model.generate_text(
            input_ids=inputs.input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            max_new_tokens=MAX_NEW_TOKENS
        )

    pred = pred.strip()
    gt_list = sample["texts"]
    #print("GT LIST", gt_list[0])

    # ---- Metrics ----
    exact = any(pred == normalize_text(str(gt)) for gt in gt_list)
    relaxed = any(relaxed_match(pred, gt) for gt in gt_list)

    if exact: exact_correct += 1
    if relaxed: relaxed_correct += 1

    pred_cells = re.split(r"[\n,\|]+", pred)
    gt_cells = [str(gt["assistant"]) for gt in gt_list]

    f1 = f1_score(
        [c for c in pred_cells if c.strip()],
        [c for c in gt_cells if c.strip()]
    )
    rms_f1_list.append(f1)

    lev = levenshtein(normalize_text(pred), normalize_text(gt_list))
    lev_list.append(lev)

    bleu_preds.append(pred)
    bleu_refs.append([normalize_text(str(gt["assistant"])) for gt in gt_list])

        # Running BLEU
    running_bleu = bleu.compute(
        predictions=bleu_preds,
        references=bleu_refs
    )["bleu"]

        # OPTIONAL: per-sample BLEU (debug)
        # sample_bleu = bleu.compute(
        #     predictions=[pred_clean],
        #     references=[[normalize_text(str(gt)) for gt in gt_list]]
        # )["bleu"]

        # ROUGE-L
    rouge_sample = max([
        rouge.score(pred, str(gt["assistant"]))["rougeL"].fmeasure
        for gt in gt_list
    ])
    rouge_scores.append(rouge_sample)

    total += 1

    # print a few samples
    if total <= 3:
        print("\n=== SAMPLE ===")
        print("PRED:\n", pred)
        print("GT:\n", gt_list)
        print(f"F1: {f1:.4f} | Lev: {lev}")
        print("ROUGE-L:", rouge_sample)
        print("Running BLEU:", running_bleu)

bleu_score = bleu.compute(predictions=bleu_preds, references=bleu_refs)

print("\n=== FINAL RESULTS ===")
print(f"Total: {total}")
print(f"Exact Acc: {exact_correct/total:.4f}")
print(f"Relaxed Acc: {relaxed_correct/total:.4f}")
print(f"RMS F1: {np.mean(rms_f1_list):.4f}")
print(f"Mean Levenshtein: {np.mean(lev_list):.4f}")
print(f"BLEU: {bleu_score['bleu']:.4f}")
print(f"ROUGE-L: {np.mean(rouge_scores):.4f}")

  0%|          | 1/2700 [00:02<2:02:53,  2.73s/it]


=== SAMPLE ===
PRED:
 a bar chart showing the percentage of social media users in the us
GT:
 [{'user': 'Please clarify the meaning conveyed by this graph.', 'assistant': 'This statistic presents the reach of the most popular social networks among female beauty consumers in the United States as of August 2016. During the survey period, 62 percent of respondents had an Instagram account.', 'source': 'chart2text-statista'}]
F1: 0.0000 | Lev: 258
ROUGE-L: 0.20833333333333331
Running BLEU: 0.0


  0%|          | 2/2700 [00:03<1:18:17,  1.74s/it]


=== SAMPLE ===
PRED:
 a graph showing the number of births in the united states
GT:
 [{'user': 'Explain what this graph is communicating.', 'assistant': "In France, the crude birth rate in 1800 was 29.4 live births per thousand people, meaning that 2.9 percent of the population had been born in that year. In the first half of the nineteenth century France's crude birth rate dropped from it's highest recorded level of 29.4 in 1800, to 21.9 by 1850. In the second half of the 1800s the crude birth rate rose again, to 25.5 in 1875, as the Second Republic and Second Empire were established, which was a time of economic prosperity and the modernization of the country. From then until 1910 there was a gradual decline, until the First World War caused a huge decline, resulting in a record low crude birth rate of 13.3 by 1920 (the figures for individual years fell even lower than this). The figure then bounced back in the early 1920s, before then falling again until the Second World War. After

  0%|          | 3/2700 [00:05<1:11:03,  1.58s/it]


=== SAMPLE ===
PRED:
 a pie chart that shows the percentage of people who have been affected by the covid - 19 pandemic
GT:
 [{'user': 'Please describe the key points or trends indicated by this graph.', 'assistant': 'Over 95 percent of Russian business managing representatives stated that the coronavirus spread in the country would lead to strong or rather strong damages to the national economy, while the share of those with positive prospects was represented by a minority.  For further information about the coronavirus (COVID-19) pandemic, please visit our dedicated Facts and Figures page.', 'source': 'chart2text-statista'}]
F1: 0.0000 | Lev: 402
ROUGE-L: 0.21052631578947367
Running BLEU: 0.0


  1%|          | 27/2700 [00:33<55:06,  1.24s/it]


KeyboardInterrupt: 